## 5 Idiom Classification

This notebook uses the decoder embeddings of the fine tuned whisper model to train an audio idiom classifier.

In [ ]:
import sys
import os
from pathlib import Path
import torch
import joblib
from transformers import WhisperProcessor, WhisperForConditionalGeneration

notebook_dir = Path.cwd()
whisper_dir = notebook_dir.parent
sys.path.append(str(whisper_dir))

from whisper_asr import load_all_data, train_classifier, extract_encoder_embeddings
from whisper_asr.utils import get_best_gpu

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device(f"cuda:{get_best_gpu()}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Load model and processor

In [ ]:
MODEL_PATH = "../models/whisper-medium-rm-all-it"
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)
print("Model loaded.")

Load train and test split

In [ ]:
train_df = load_all_data("train")
test_df = load_all_data("test")

Extract encoder embeddings for train and test set

In [ ]:
print("Extracting train encoder embeddings...")
train_embeddings = extract_encoder_embeddings(
    model, processor,
    train_df["audio_path"].tolist(),
    device=DEVICE, batch_size=8
)
print("Extracting test encoder embeddings...")
test_embeddings = extract_encoder_embeddings(
    model, processor,
    test_df["audio_path"].tolist(),
    device=DEVICE, batch_size=8
)

Train classifier

In [ ]:
train_labels = train_df["idiom"].tolist()
test_labels = test_df["idiom"].tolist()

classifier, predictions = train_classifier(
    train_embeddings, train_labels,
    test_embeddings, test_labels
)

joblib.dump(classifier, "../models/idiom_classifier.pkl")